In [ ]:
# Instalação das dependências necessárias
!apt-get install -y fluidsynth build-essential libasound2-dev libjack-dev > /dev/null
!pip install music21 pyfluidsynth midiutil > /dev/null

In [ ]:
# Import das bibliotecas necessárias
import music21
import random
import time
import os
from IPython.display import Audio, display, HTML

In [ ]:
# Código da Gramática Generativa Não-Determinística
class Gramatica:
    def __init__(self):
        # Regras de produção contendo todas as variações estruturais, instrumentais, harmônicas e melódicas
        self.regras = {
            # 1. Estruturas Gerais (REFRÃO/VERSO/PONTE)
            'S': [
                ['INTRO_SECAO', 'VERSO_SECAO', 'PRE_SECAO', 'REFRAO_SECAO', 'VERSO_SECAO', 'PRE_SECAO', 'REFRAO_SECAO', 'PONTE_SECAO', 'REFRAO_SECAO', 'OUTRO_SECAO'],
                ['REFRAO_SECAO', 'VERSO_SECAO', 'PRE_SECAO', 'REFRAO_SECAO', 'VERSO_SECAO', 'PRE_SECAO', 'REFRAO_SECAO', 'PONTE_SECAO', 'REFRAO_SECAO', 'OUTRO_SECAO'],
                ['INTRO_SECAO', 'VERSO_SECAO', 'REFRAO_SECAO', 'VERSO_SECAO', 'REFRAO_SECAO', 'OUTRO_SECAO'],
                ['INTRO_SECAO', 'VERSO_SECAO', 'PRE_SECAO', 'REFRAO_SECAO', 'VERSO_SECAO', 'PRE_SECAO', 'REFRAO_SECAO', 'PONTE_SECAO', 'REFRAO_SECAO', 'REFRAO_SECAO', 'OUTRO_SECAO'],
                ['INTRO_SECAO', 'VERSO_SECAO', 'PRE_SECAO', 'REFRAO_SECAO', 'VERSO_SECAO', 'PRE_SECAO', 'REFRAO_SECAO', 'OUTRO_SECAO']
            ],

            # 2. Definição das Seções com Climas Rítmicos e com Símbolos Harmônicos e Melódicos
            'INTRO_SECAO': [['INTRO_PAD_ESTILO', 'INTRO_BAIXO_ESTILO', 'INTRO_GUIT_ESTILO', 'INTRO_BATERA_ESTILO', 'INTRO_COMPASSOS', 'INTRO_HARMONIA', 'INTRO_MELODIA']],
            'VERSO_SECAO': [['VERSO_PAD_ESTILO', 'VERSO_BAIXO_ESTILO', 'VERSO_GUIT_ESTILO', 'VERSO_BATERA_ESTILO', 'VERSO_COMPASSOS', 'VERSO_HARMONIA', 'VERSO_MELODIA']],
            'PRE_SECAO':   [['PRE_PAD_ESTILO', 'PRE_BAIXO_ESTILO', 'PRE_GUIT_ESTILO', 'PRE_BATERA_ESTILO', 'PRE_COMPASSOS', 'PRE_HARMONIA', 'PRE_MELODIA']],
            'REFRAO_SECAO':[['REFRAO_PAD_ESTILO', 'REFRAO_BAIXO_ESTILO', 'REFRAO_GUIT_ESTILO', 'REFRAO_BATERA_ESTILO', 'REFRAO_COMPASSOS', 'REFRAO_HARMONIA', 'REFRAO_MELODIA']],
            'PONTE_SECAO': [['PONTE_PAD_ESTILO', 'PONTE_BAIXO_ESTILO', 'PONTE_GUIT_ESTILO', 'PONTE_BATERA_ESTILO', 'PONTE_COMPASSOS', 'PONTE_HARMONIA', 'PONTE_MELODIA']],
            'OUTRO_SECAO': [['OUTRO_PAD_ESTILO', 'OUTRO_BAIXO_ESTILO', 'OUTRO_GUIT_ESTILO', 'OUTRO_BATERA_ESTILO', 'OUTRO_COMPASSOS', 'OUTRO_HARMONIA', 'OUTRO_MELODIA']],

            # 3. Estilos de Pads de Cordas por Seção
            'INTRO_PAD_ESTILO': [['STR_SUSTENTADO'], ['STR_PULSANTE']],
            'VERSO_PAD_ESTILO': [['STR_SUSTENTADO'], ['STR_SINCOPADO']],
            'PRE_PAD_ESTILO':   [['STR_PULSANTE'], ['STR_SINCOPADO']],
            'REFRAO_PAD_ESTILO':[['STR_SUSTENTADO'], ['STR_PULSANTE'], ['STR_SINCOPADO']],
            'PONTE_PAD_ESTILO': [['STR_SUSTENTADO'], ['STR_SINCOPADO']],
            'OUTRO_PAD_ESTILO': [['STR_SUSTENTADO']],

            # 4. Estilos de Baixo por Seção
            'INTRO_BAIXO_ESTILO': [['BASS_CONTINUO'], ['BASS_RESPIRO']],
            'VERSO_BAIXO_ESTILO': [['BASS_CONTINUO'], ['BASS_MARCADO']],
            'PRE_BAIXO_ESTILO':   [['BASS_MARCADO'], ['BASS_OITAVADO']],
            'REFRAO_BAIXO_ESTILO':[['BASS_MARCADO'], ['BASS_OITAVADO'], ['BASS_RESPIRO']],
            'PONTE_BAIXO_ESTILO': [['BASS_CONTINUO'], ['BASS_OITAVADO']],
            'OUTRO_BAIXO_ESTILO': [['BASS_RESPIRO']],

            # 5. Estilos de Guitarra por Seção
            'INTRO_GUIT_ESTILO': [['GUIT_REGULAR'], ['GUIT_ESPARSO']],
            'VERSO_GUIT_ESTILO': [['GUIT_REGULAR'], ['GUIT_SINCOPADO'], ['GUIT_ESPARSO']],
            'PRE_GUIT_ESTILO':   [['GUIT_SINCOPADO'], ['GUIT_TERCINAS']],
            'REFRAO_GUIT_ESTILO':[['GUIT_REGULAR'], ['GUIT_SINCOPADO'], ['GUIT_TERCINAS']],
            'PONTE_GUIT_ESTILO': [['GUIT_BLOCOS'], ['GUIT_ARPEJO_RAPIDO'], ['GUIT_PULSACAO_SINC'], ['GUIT_TERCINAS_DRAMATICAS']],
            'OUTRO_GUIT_ESTILO': [['GUIT_ESPARSO']],

            # 6. Estilos de Bateria por Seção
            'INTRO_BATERA_ESTILO': [['BAT_VERSO'], ['BAT_SILENCIO']],
            'VERSO_BATERA_ESTILO': [['BAT_VERSO'], ['BAT_SILENCIO']],
            'PRE_BATERA_ESTILO':   [['BAT_SILENCIO'], ['BAT_STANDARD']],
            'REFRAO_BATERA_ESTILO':[['BAT_STANDARD'], ['BAT_HIP_HOP'], ['BAT_TRIPLETS'], ['BAT_MINIMAL']],
            'PONTE_BATERA_ESTILO': [['BAT_STANDARD'], ['BAT_HIP_HOP'], ['BAT_TRIPLETS'], ['BAT_MINIMAL']],
            'OUTRO_BATERA_ESTILO': [['BAT_SILENCIO']],

            # 7. Tamanho dos Blocos em Compassos
            'INTRO_COMPASSOS': [['COMPASSOS_2']],
            'VERSO_COMPASSOS': [['COMPASSOS_4']],
            'PRE_COMPASSOS':   [['COMPASSOS_2']],
            'REFRAO_COMPASSOS':[['COMPASSOS_4']],
            'PONTE_COMPASSOS': [['COMPASSOS_4']],
            'OUTRO_COMPASSOS': [['COMPASSOS_2']],

            # 8. Regras de Harmonias
            'INTRO_HARMONIA': [
                ['HARM_A2_E3_C4__F2_C3_A3'],
                ['HARM_D2_A3_F4__E2_B3_G#4'],
                ['HARM_A2_E3_G3_C4__F2_C3_G3_A3'],
                ['HARM_F#2_C#3_A3__D2_A3_F#4']
            ],
            'VERSO_HARMONIA': [
                ['HARM_A2_E3_C4__F2_C3_A3__C2_G3_E4__G2_D3_B3'],
                ['HARM_A2_E3_C4__D2_A3_F4__F2_C3_A3__E2_B3_G#4'],
                ['HARM_A2_E3_G3_C4__F2_C3_E3_A3__D2_A3_C4_F4__E2_B3_D4_G#4'],
                ['HARM_A2_E3_G3_B3__F2_C3_G3_A3__D2_A3_C4_E4__E2_B3_D4_G#4'],
                ['HARM_A2_E3_C4__G2_D3_B3__F2_C3_A3__F2_C3_G3_B3']
            ],
            'PRE_HARMONIA': [
                ['HARM_F2_C3_A3__G2_D3_B3'],
                ['HARM_D2_A3_F4__E2_B3_G#4'],
                ['HARM_F2_C3_G3_A3__E2_B3_D4_G#4'],
                ['HARM_B1_F2_A2_D3__E2_B2_D3_G#3']
            ],
            'REFRAO_HARMONIA': [
                ['HARM_F2_C3_A3__C2_G3_E4__G2_D3_B3__A2_E3_C4'],
                ['HARM_D2_A3_F4__F2_C3_A3__C2_G3_E4__G2_D3_B3'],
                ['HARM_F2_C3_E3_A3__C2_G3_D4_E4__G2_D3_F3_B3__A2_E3_G3_C4'],
                ['HARM_D2_A3_C4_F4__F2_C3_E3_A3__C2_G3_D4_E4__E2_B3_D4_G#4']
            ],
            'PONTE_HARMONIA': [
                ['HARM_D2_A3_C4_F4__E2_B3_D4_G#4__F2_C3_E3_A3__A2_E3_G3_C4'],
                ['HARM_F#2_C#3_A#3__D2_A3_F4__E2_B3_G#4__A2_E3_C4'],
                ['HARM_F2_C3_E3_G3_A3__D2_A3_C4_E4_F4__B1_F2_A2_D3__E2_B2_D3_G#3'],
                ['HARM_G#2_D#3_B3__F#2_C#3_A#3__E2_B3_G#4__D#2_A#3_F##4']
            ],
            'OUTRO_HARMONIA': [
                ['HARM_A2_E3_C4__F2_C3_A3__D2_A3_F4__A1_E2_A3'],
                ['HARM_F2_C3_A3__G2_D3_B3__A2_E3_G3_C4__A1_E2_A2'],
                ['HARM_D2_A3_C4_F4__E2_B3_D4_G#4__A2_E3_C4__A1_E2_A2'],
                ['HARM_F2_C3_G3_A3__G2_D3_F3_B3__A2_E3_C4__A1_E2_A3']
            ],

            # 9. Regras de Melodias
            'INTRO_MELODIA': [
                ['MEL_A3_E3_A3_B3_C4_B3_A3_E3'],
                ['MEL_E3_G3_A3_B3_D4_C4_B3_A3'],
                ['MEL_C4_B3_A3_E3_F3_E3_C4_B3']
            ],
            'VERSO_MELODIA': [
                ['MEL_A3_B3_C4_E4_D4_C4_B3_A3'],
                ['MEL_E3_A3_B3_C4_E4_G4_F4_E4'],
                ['MEL_C4_B3_A3_G3_F3_A3_C4_E4'],
                ['MEL_A3_C4_D4_E4_G4_E4_D4_C4']
            ],
            'PRE_MELODIA': [
                ['MEL_D4_E4_F4_A4_G4_F4_E4_D4'],
                ['MEL_B3_D4_E4_G#4_B4_A4_G#4_E4'],
                ['MEL_F3_A3_D4_F4_E4_D4_C4_B3']
            ],
            'REFRAO_MELODIA': [
                ['MEL_E4_G4_A4_C5_B4_A4_G4_E4'],
                ['MEL_A4_B4_C5_E5_D5_C5_B4_A4'],
                ['MEL_C4_D4_E4_G4_A4_E4_G4_A4'],
                ['MEL_E4_F#4_G4_A4_B4_C5_D5_E5']
            ],
            'PONTE_MELODIA': [
                ['MEL_D4_F4_G#4_A4_B4_D5_C5_A4'],
                ['MEL_E4_G#4_B4_C5_D5_F5_E5_B4'],
                ['MEL_A4_B4_C5_E5_F5_D5_E5_C5']
            ],
            'OUTRO_MELODIA': [
                ['MEL_C4_B3_A3_E3_A3_rest'],
                ['MEL_E4_D4_C4_B3_A3_G3_A3_rest'],
                ['MEL_A3_E3_C4_B3_A3_rest_rest_rest']
            ]
        }
    # Função de Expansão da Gramática
    def expandir(self, symbol):
        if symbol not in self.regras:
            return [symbol]
        chosen_expansion = random.choice(self.regras[symbol])
        final_expansion = []
        for sym in chosen_expansion:
            final_expansion.extend(self.expandir(sym))
        return final_expansion

In [ ]:
def configurar_transposicao():
    #Sorteia a transposição musical e retorna o intervalo de transposição configurado.
    intervalos_possiveis = ['m2', 'M2', 'm3', 'M3', 'p4', 'd5', 'p5', 'm6', 'M6', 'm7', 'M7', 'P1']
    intervalo_transposicao = random.choice(intervalos_possiveis)
    direcao = random.choice(['up', 'down'])
    transpose_interval = music21.interval.Interval(intervalo_transposicao)
    if direcao == 'down' and intervalo_transposicao != 'P1':
        transpose_interval = transpose_interval.reverse()
    return transpose_interval, direcao

def aplicar_transposicao(elemento, transpose_interval):
    if elemento is None:
        return None
    return elemento.transpose(transpose_interval)

def gerar_estrutura_arranjo(gramatica_instancia):
    # Faz a expansão da gramática e o mapeamento das seções.
    estrutura_expandida = gramatica_instancia.expandir('S')

    # Agrupa os símbolos expandidos de 7 em 7 (Pad, Baixo, Guit, Batera, Compassos, Harmonia, Melodia)
    secoes_processadas = []
    for i in range(0, len(estrutura_expandida), 7):
        secoes_processadas.append(estrutura_expandida[i:i+7])

    target_len = len(secoes_processadas)
    secoes_nomes = []

    for opcao_s in gramatica_instancia.regras['S']:
        if len(opcao_s) == target_len:
            secoes_nomes = [x.split('_')[0] for x in opcao_s]
            break

    return secoes_processadas, secoes_nomes

In [ ]:
def criar_score(secoes_processadas, secoes_nomes, tempo_bpm, transpose_interval):
    # Cria a estrutura de score e as faixas instrumentais do music21.
    score = music21.stream.Score()
    score.append(music21.tempo.MetronomeMark(number=tempo_bpm))

    part_voz = music21.stream.Part()
    part_strings = music21.stream.Part()
    part_guitarra = music21.stream.Part()
    part_baixo = music21.stream.Part()
    part_bateria = music21.stream.Part()

    for idx, sec in enumerate(secoes_processadas):
        pad_estilo, baixo_estilo, guit_estilo, batera_estilo, compassos_estilo, harmonia_estilo, melodia_estilo = sec
        tipo_secao = secoes_nomes[idx]
        compassos = int(compassos_estilo.split('_')[1])

        # Decodifica as harmonias e melodias diretamente dos terminais gerados pela gramática
        acordes_pool = [acorde.replace('__', '_') for acorde in harmonia_estilo.replace('HARM_', '').split('__')]
        melodia_pool = melodia_estilo.replace('MEL_', '').split('_')

        for c in range(compassos):
            acorde_notas = acordes_pool[c % len(acordes_pool)].split('_')

            # 1. Pads de Cordas
            if pad_estilo == 'STR_SUSTENTADO':
                raw_chord = music21.chord.Chord(acorde_notas, quarterLength=4.0)
                part_strings.append(aplicar_transposicao(raw_chord, transpose_interval))
            elif pad_estilo == 'STR_PULSANTE':
                for _ in range(4):
                    raw_chord = music21.chord.Chord(acorde_notas, quarterLength=1.0)
                    part_strings.append(aplicar_transposicao(raw_chord, transpose_interval))
            elif pad_estilo == 'STR_SINCOPADO':
                c1 = music21.chord.Chord(acorde_notas, quarterLength=1.5)
                c2 = music21.chord.Chord(acorde_notas, quarterLength=1.5)
                c3 = music21.chord.Chord(acorde_notas, quarterLength=1.0)
                part_strings.append(aplicar_transposicao(c1, transpose_interval))
                part_strings.append(aplicar_transposicao(c2, transpose_interval))
                part_strings.append(aplicar_transposicao(c3, transpose_interval))

            # 2. Baixo
            if baixo_estilo == 'BASS_CONTINUO':
                raw_bass = music21.note.Note(acorde_notas[0], quarterLength=4.0)
                part_baixo.append(aplicar_transposicao(raw_bass, transpose_interval))
            elif baixo_estilo == 'BASS_MARCADO':
                for _ in range(4):
                    raw_bass = music21.note.Note(acorde_notas[0], quarterLength=1.0)
                    part_baixo.append(aplicar_transposicao(raw_bass, transpose_interval))
            elif baixo_estilo == 'BASS_OITAVADO':
                b_pitch = music21.pitch.Pitch(acorde_notas[0])
                b_pitch_oct = music21.pitch.Pitch(acorde_notas[0])
                b_pitch_oct.octave += 1
                n1 = music21.note.Note(b_pitch, quarterLength=1.5)
                n2 = music21.note.Note(b_pitch_oct, quarterLength=1.5)
                n3 = music21.note.Note(b_pitch, quarterLength=1.0)
                part_baixo.append(aplicar_transposicao(n1, transpose_interval))
                part_baixo.append(aplicar_transposicao(n2, transpose_interval))
                part_baixo.append(aplicar_transposicao(n3, transpose_interval))
            elif baixo_estilo == 'BASS_RESPIRO':
                raw_bass = music21.note.Note(acorde_notas[0], quarterLength=3.0)
                part_baixo.append(aplicar_transposicao(raw_bass, transpose_interval))
                part_baixo.append(music21.note.Rest(quarterLength=1.0))

            # 3. Guitarra / Acompanhamento Harmônico
            if guit_estilo == 'GUIT_REGULAR':
                for i in range(8):
                    p = music21.pitch.Pitch(acorde_notas[i % len(acorde_notas)])
                    p.octave += 1
                    raw_note = music21.note.Note(p, quarterLength=0.5)
                    part_guitarra.append(aplicar_transposicao(raw_note, transpose_interval))
            elif guit_estilo == 'GUIT_SINCOPADO':
                for i in range(4):
                    p = music21.pitch.Pitch(acorde_notas[i % len(acorde_notas)])
                    p.octave += 1
                    raw_note = music21.note.Note(p, quarterLength=0.75)
                    part_guitarra.append(aplicar_transposicao(raw_note, transpose_interval))
                part_guitarra.append(music21.note.Rest(quarterLength=1.0))
            elif guit_estilo == 'GUIT_TERCINAS':
                for i in range(6):
                    p = music21.pitch.Pitch(acorde_notas[i % len(acorde_notas)])
                    p.octave += 1
                    raw_note = music21.note.Note(p, quarterLength=0.66)
                    part_guitarra.append(aplicar_transposicao(raw_note, transpose_interval))
                part_guitarra.append(music21.note.Rest(quarterLength=0.04))
            elif guit_estilo == 'GUIT_ESPARSO':
                part_guitarra.append(aplicar_transposicao(music21.chord.Chord(acorde_notas, quarterLength=4.0), transpose_interval))
            elif guit_estilo == 'GUIT_BLOCOS':
                g_chord = music21.chord.Chord(acorde_notas, quarterLength=4.0)
                for p in g_chord.pitches:
                    p.octave += 1
                part_guitarra.append(aplicar_transposicao(g_chord, transpose_interval))
            elif guit_estilo == 'GUIT_ARPEJO_RAPIDO':
                for i in range(16):
                    p = music21.pitch.Pitch(acorde_notas[i % len(acorde_notas)])
                    p.octave += 1
                    raw_note = music21.note.Note(p, quarterLength=0.25)
                    part_guitarra.append(aplicar_transposicao(raw_note, transpose_interval))
            elif guit_estilo == 'GUIT_PULSACAO_SINC':
                for _ in range(4):
                    g_chord = music21.chord.Chord(acorde_notas, quarterLength=0.75)
                    for p in g_chord.pitches:
                        p.octave += 1
                    part_guitarra.append(aplicar_transposicao(g_chord, transpose_interval))
                part_guitarra.append(music21.note.Rest(quarterLength=1.0))
            elif guit_estilo == 'GUIT_TERCINAS_DRAMATICAS':
                for i in range(6):
                    p = music21.pitch.Pitch(acorde_notas[i % len(acorde_notas)])
                    p.octave += 1
                    raw_note = music21.note.Note(p, quarterLength=0.66)
                    part_guitarra.append(aplicar_transposicao(raw_note, transpose_interval))
                part_guitarra.append(music21.note.Rest(quarterLength=0.04))

            # 4. Voz / Linha Melódica
            tempo_restante = 4.0
            while tempo_restante > 0:
                pitch_suave = random.choice(melodia_pool)
                duracao = random.choice([2.0, 1.0, 0.5, 0.5, 1.5])
                if duracao > tempo_restante:
                    duracao = tempo_restante
                if pitch_suave == 'rest':
                    part_voz.append(music21.note.Rest(quarterLength=duracao))
                else:
                    raw_note = music21.note.Note(pitch_suave, quarterLength=duracao)
                    part_voz.append(aplicar_transposicao(raw_note, transpose_interval))
                tempo_restante -= duracao

            # 5. Bateria
            if batera_estilo == 'BAT_STANDARD':
                b1 = music21.note.Note('C2', quarterLength=1.0)
                b2 = music21.note.Note('D2', quarterLength=1.0)
                b3 = music21.note.Note('C2', quarterLength=1.0)
                b4 = music21.note.Note('D2', quarterLength=1.0)
                part_bateria.append([b1, b2, b3, b4])
            elif batera_estilo == 'BAT_HIP_HOP':
                b1 = music21.note.Note('C2', quarterLength=0.75)
                b2 = music21.note.Note('D2', quarterLength=1.25)
                b3 = music21.note.Note('C2', quarterLength=0.5)
                b4 = music21.note.Note('D2', quarterLength=1.5)
                part_bateria.append([b1, b2, b3, b4])
            elif batera_estilo == 'BAT_TRIPLETS':
                b1 = music21.note.Note('C2', quarterLength=1.33)
                b2 = music21.note.Note('D2', quarterLength=1.33)
                b3 = music21.note.Note('C2', quarterLength=1.34)
                part_bateria.append([b1, b2, b3])
            elif batera_estilo == 'BAT_MINIMAL':
                b1 = music21.note.Note('C2', quarterLength=1.5)
                b2 = music21.note.Note('D2', quarterLength=2.5)
                part_bateria.append([b1, b2])
            elif batera_estilo == 'BAT_VERSO':
                b_verso = music21.note.Note('C2', quarterLength=1.0)
                r_verso = music21.note.Rest(quarterLength=3.0)
                part_bateria.append([b_verso, r_verso])
            elif batera_estilo == 'BAT_SILENCIO':
                part_bateria.append(music21.note.Rest(quarterLength=4.0))

    # Montagem final do Score
    score.insert(0, part_voz)
    score.insert(0, part_strings)
    score.insert(0, part_guitarra)
    score.insert(0, part_baixo)
    score.insert(0, part_bateria)

    return score

In [ ]:
# Execução do módulo de arranjo
gramatica = Gramatica()

# Lista para guardar informações das músicas geradas
musicas_geradas = []

for i in range(1, 4):
    # Forçar uma nova semente baseada no tempo para garantir variação
    random.seed(time.time_ns() + i)

    # Configurações do arranjo
    tempo_bpm = random.randint(62, 78)
    transpose_interval, direcao = configurar_transposicao()
    secoes_processadas, secoes_nomes = gerar_estrutura_arranjo(gramatica)

    # Criar o Score
    score = criar_score(secoes_processadas, secoes_nomes, tempo_bpm, transpose_interval)

    nome_arquivo_midi = f"musica_{i}.mid"
    score.write("midi", fp=nome_arquivo_midi)

    musicas_geradas.append({
        "id": i,
        "midi": nome_arquivo_midi,
        "wav": f"musica_{i}.wav",
        "bpm": tempo_bpm,
        "transposicao": f"{transpose_interval.niceName} ({direcao})",
        "estrutura": " -> ".join(secoes_nomes)
    })

    print(f"Música {i} Gerada com Sucesso:")
    print(f"  - Arquivo: {nome_arquivo_midi}")
    print(f"  - Estrutura: {' -> '.join(secoes_nomes)}")
    print(f"  - BPM: {tempo_bpm} | Transposição: {transpose_interval.niceName} ({direcao})\n")

In [ ]:
# Renderização de Áudio para as Três Músicas Geradas
sf2_path = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
if not os.path.exists(sf2_path):
    sf2_path = "/usr/share/sounds/sf2/TimGM6mb.sf2"

for m in musicas_geradas:
    display(HTML(f"<h3>Música {m['id']}</h3>"))
    display(HTML(f"<p><b>Andamento:</b> {m['bpm']} BPM | <b>Transposição:</b> {m['transposicao']}</p>"))
    display(HTML(f"<p><b>Caminho do MIDI:</b> {m['midi']} | <b>Estrutura:</b> {m['estrutura']}</p>"))

    # Sintetizar MIDI para WAV usando o FluidSynth
    os.system(f"fluidsynth -ni '{sf2_path}' {m['midi']} -F {m['wav']} -r 44100 > /dev/null")

    # Exibir o player de áudio para cada uma delas
    display(Audio(m['wav']))
    display(HTML("<hr>"))